# SPAI — Learnable Frequency-Masking Radius: Kaggle Smoke Test

This notebook fine-tunes SPAI (CVPR 2025) on a tiny medical dataset to verify that the
**parameterized (learnable) masking radius** works end-to-end:

1. Clone the thesis repository and install dependencies.
2. Download the author's released `spai.pth` checkpoint (natural-image weights).
3. Build the dataset CSV from `medical_spai_dataset/` (41 train / 8 val images).
4. **Run A (baseline)** — fine-tune with the original fixed radius `r = 16`.
5. **Run B (ours)** — fine-tune with the learnable radius, watch `r` move during training.
6. Evaluate both checkpoints on the validation split and print the radius trajectory.

> ⚠️ This is a **smoke test**: with only 49 images the metrics are not meaningful.
> The success criteria are: training runs without errors, the radius receives gradients
> and moves away from 16, checkpoints save/load, and `test` produces metrics.

**Kaggle setup:** GPU accelerator **T4 x2** (only GPU 0 is used — the codebase is single-GPU).

## 1. Clone the repository and install dependencies

In [ ]:
%cd /kaggle/working
!rm -rf spai-parameterized-radius
!git clone https://github.com/Kashshaf-Labib/spai-parameterized-radius.git
%cd /kaggle/working/spai-parameterized-radius
!pip install -q -r requirements-kaggle.txt


In [ ]:
# Environment: no Neptune account on Kaggle, single GPU, quieter tokenizer forks.
import os
os.environ["DISABLE_NEPTUNE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")


## 2. Download the released SPAI checkpoint (fine-tuning starting point)

In [ ]:
!mkdir -p weights
!gdown 1vvXmZqs6TVJdj8iF1oJ4L_fcgdQrp_YI -O weights/spai.pth
!ls -lh weights


## 3. Build the dataset CSV (columns: `image,class,split`)

In [ ]:
!mkdir -p datasets
!python -m spai.tools.create_dir_csv \
  --train_dir medical_spai_dataset/train \
  --val_dir medical_spai_dataset/val \
  -o datasets/medical_smoke.csv \
  -r .

import pandas as pd
df = pd.read_csv("datasets/medical_smoke.csv")
print(df.groupby(["split", "class"]).size())
df.head()


## 4. Run A — baseline fine-tune with the FIXED radius (r = 16)

Uses the original `configs/spai.yaml`. `--finetune-from` initializes the full model from
`spai.pth`. Short run (2 epochs) — it only proves the fine-tuning path works and gives a
baseline to compare against.

In [ ]:
!python -m spai train \
  --cfg configs/spai.yaml \
  --batch-size 4 \
  --data-path datasets/medical_smoke.csv \
  --csv-root-dir . \
  --finetune-from weights/spai.pth \
  --output output/fixed_radius \
  --tag smoke \
  --amp-opt-level O0 \
  --data-workers 2 \
  --save-all \
  --opt TRAIN.EPOCHS 2 \
  --opt TRAIN.WARMUP_EPOCHS 1 \
  --opt PRINT_FREQ 5 \
  --opt MODEL.FEATURE_EXTRACTION_BATCH 128 \
  --opt DATA.TEST_PREFETCH_FACTOR 1


## 5. Run B — fine-tune with the LEARNABLE radius

Uses `configs/spai_learnable_radius.yaml` (`MODEL.FRE.LEARNABLE_MASKING_RADIUS: True`).
The radius parameter starts at 16 and is trained with its own learning rate
(`TRAIN.RADIUS_LR`, raised to 0.05 here so the movement is clearly visible within the
~50 optimizer steps of this tiny run; for full-scale training keep the default 0.01).

Watch the `masking_radius` value in the per-iteration log lines and the
`Masking radius | Epoch N | r = ...` lines.

In [ ]:
!python -m spai train \
  --cfg configs/spai_learnable_radius.yaml \
  --batch-size 4 \
  --data-path datasets/medical_smoke.csv \
  --csv-root-dir . \
  --finetune-from weights/spai.pth \
  --output output/learnable_radius \
  --tag smoke \
  --amp-opt-level O0 \
  --data-workers 2 \
  --save-all \
  --opt TRAIN.EPOCHS 5 \
  --opt TRAIN.WARMUP_EPOCHS 1 \
  --opt TRAIN.RADIUS_LR 0.05 \
  --opt PRINT_FREQ 5 \
  --opt MODEL.FEATURE_EXTRACTION_BATCH 128 \
  --opt DATA.TEST_PREFETCH_FACTOR 1


## 6. Radius trajectory — did the radius actually learn?

In [ ]:
import re, glob

log_files = sorted(glob.glob("output/learnable_radius/finetune/smoke/log_rank*.txt"))
trajectory = []
for log_file in log_files:
    with open(log_file) as f:
        for line in f:
            m = re.search(r"Masking radius \| Epoch (\d+) \| r = ([0-9.]+)", line)
            if m:
                trajectory.append((int(m.group(1)), float(m.group(2))))

print("Per-epoch masking radius (started at 16.0):")
for epoch, r in trajectory:
    print(f"  epoch {epoch}: r = {r:.4f}")

if trajectory and abs(trajectory[-1][1] - 16.0) > 1e-3:
    print("\n✅ SMOKE TEST PASSED: the radius moved — gradients flow and it is being learned.")
elif trajectory:
    print("\n⚠️ Radius barely moved. Check the training log for the masking_radius values "
          "in the per-iteration lines; with a tiny dataset small movement can be normal.")
else:
    print("\n❌ No radius log lines found — check the training log above for errors.")


## 7. Evaluate both runs on the validation split

In [ ]:
print("=" * 30, "FIXED RADIUS (baseline)", "=" * 30)
!python -m spai test \
  --cfg configs/spai.yaml \
  --batch-size 1 \
  --model output/fixed_radius/finetune/smoke \
  --output output/test_fixed \
  --tag smoke_fixed \
  --split val \
  --test-csv datasets/medical_smoke.csv \
  --test-csv-root-dir . \
  --opt MODEL.PATCH_VIT.MINIMUM_PATCHES 4 \
  --opt DATA.NUM_WORKERS 2 \
  --opt MODEL.FEATURE_EXTRACTION_BATCH 128 \
  --opt DATA.TEST_PREFETCH_FACTOR 1


In [ ]:
print("=" * 30, "LEARNABLE RADIUS (ours)", "=" * 30)
!python -m spai test \
  --cfg configs/spai_learnable_radius.yaml \
  --batch-size 1 \
  --model output/learnable_radius/finetune/smoke \
  --output output/test_learnable \
  --tag smoke_learnable \
  --split val \
  --test-csv datasets/medical_smoke.csv \
  --test-csv-root-dir . \
  --opt MODEL.PATCH_VIT.MINIMUM_PATCHES 4 \
  --opt DATA.NUM_WORKERS 2 \
  --opt MODEL.FEATURE_EXTRACTION_BATCH 128 \
  --opt DATA.TEST_PREFETCH_FACTOR 1


In [ ]:
# Read the learned radius directly from the saved checkpoints.
import glob, torch

for ckpt_path in sorted(glob.glob("output/learnable_radius/finetune/smoke/ckpt_epoch_*.pth")):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    r = ckpt["model"]["mfvit.soft_frequency_mask.radius"].item()
    print(f"{ckpt_path}: learned radius = {r:.4f}")


## Next steps (full experiment)

Once this smoke test passes, repeat with the real dataset:

1. Replace `medical_spai_dataset/` with the full medical dataset (same
   `train|val / 0_real|1_fake` layout) or upload it as a Kaggle Dataset and point
   `create_dir_csv` at it.
2. Use the default `TRAIN.RADIUS_LR 0.01`, `TRAIN.EPOCHS 10–35`, `WARMUP_EPOCHS` scaled
   accordingly, and add a held-out `test` split (`--test_dir`) for honest evaluation.
3. Always run **both** configs (fixed vs. learnable) on the identical split — that pair is
   the ablation for the thesis, and report where the radius converges per run.
4. If GPU memory allows, raise `--batch-size`; otherwise use `--accumulation-steps`.